# Query Multilayer Networks with the DSL

This notebook demonstrates py3plex's SQL-like Domain-Specific Language (DSL) for querying multilayer networks.

## Why Use the DSL?

* **Graph-aware**: Understands multilayer structures, layers, and (node, layer) tuple semantics
* **Type-safe**: Builder API with IDE autocompletion
* **Integrated**: Compute centrality and metrics directly in queries
* **Flexible**: String syntax for quick prototyping, builder API for production

## Installation

In [ ]:
!pip install py3plex -q

## Setup: Create a Sample Network

Let's create a multilayer network with social and work layers.

In [ ]:
from py3plex.core import multinet

# Create sample network
network = multinet.multi_layer_network()

# Add edges across layers
network.add_edges([
    # Social layer
    ['Alice', 'social', 'Bob', 'social', 1],
    ['Bob', 'social', 'Charlie', 'social', 1],
    ['Charlie', 'social', 'David', 'social', 1],
    ['Alice', 'social', 'Eve', 'social', 1],
    ['David', 'social', 'Frank', 'social', 1],
    # Work layer
    ['Alice', 'work', 'Bob', 'work', 1],
    ['Bob', 'work', 'Charlie', 'work', 1],
    ['Charlie', 'work', 'Frank', 'work', 1],
    # Hobby layer
    ['Alice', 'hobby', 'David', 'hobby', 1],
    ['Eve', 'hobby', 'Frank', 'hobby', 1],
], input_type="list")

print("Network created:")
network.basic_stats()

## Method 1: String Syntax (Quick and Readable)

Use SQL-like strings for quick queries.

In [ ]:
from py3plex.dsl import execute_query

# Get all nodes
result = execute_query(network, 'SELECT nodes')
print(f"Found {len(result)} nodes")

# Get nodes from social layer only
result = execute_query(network, 'SELECT nodes FROM layer="social"')
print(f"\nSocial layer has {len(result)} nodes")

# Filter by degree
result = execute_query(network, 'SELECT nodes WHERE degree > 1')
df = result.to_pandas()
print("\nNodes with degree > 1:")
print(df)

## Method 2: Builder API (Type-Safe, Chainable)

Use the Python builder API for type safety and IDE autocompletion.

In [ ]:
from py3plex.dsl import Q, L

# Simple query
result = Q.nodes().execute(network)
print(f"Total nodes: {len(result)}")

# Query specific layer
result = (
    Q.nodes()
     .from_layers(L["social"])
     .execute(network)
)
print(f"\nSocial layer nodes: {len(result)}")

# Filter and compute
result = (
    Q.nodes()
     .from_layers(L["*"])  # All layers
     .where(degree__gt=1)  # Django-style filter
     .compute("degree", "betweenness_centrality")
     .order_by("-degree")  # Sort descending
     .execute(network)
)

df = result.to_pandas()
print("\nHigh-degree nodes with centrality:")
print(df)

## Layer Algebra: Combine Layers

Use operators to combine layers: `+` (union), `&` (intersection), `-` (difference)

In [ ]:
# Union: nodes from social OR work
result = (
    Q.nodes()
     .from_layers(L["social"] + L["work"])
     .execute(network)
)
print(f"Nodes in social + work: {len(result)}")

# All layers
result = (
    Q.nodes()
     .from_layers(L["*"])
     .execute(network)
)
print(f"Nodes in all layers: {len(result)}")

## Compute Multiple Metrics

Calculate various centrality measures in a single query.

In [ ]:
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute(
         "degree",
         "betweenness_centrality",
         "closeness_centrality",
         "pagerank"
     )
     .order_by("-betweenness_centrality")
     .limit(10)
     .execute(network)
)

df = result.to_pandas()
print("Top 10 nodes by betweenness centrality:")
print(df)

## Filter with Complex Conditions

Use Django-style lookups for flexible filtering.

In [ ]:
# Nodes with degree greater than 1 in the social layer
result = (
    Q.nodes()
     .from_layers(L["social"])
     .where(degree__gt=1)
     .compute("degree")
     .execute(network)
)

df = result.to_pandas()
print("Social layer hubs (degree > 1):")
print(df)

# Nodes NOT in a specific layer
result = (
    Q.nodes()
     .where(layer__ne="hobby")
     .compute("degree")
     .execute(network)
)

df = result.to_pandas()
print(f"\nNodes not in hobby layer: {len(df)}")

## Query Edges

Query edges within or between layers.

In [ ]:
# Get all edges
result = Q.edges().execute(network)
print(f"Total edges: {len(result)}")

# Edges from social layer only
result = (
    Q.edges()
     .from_layers(L["social"])
     .execute(network)
)
print(f"\nEdges in social layer: {len(result)}")

# View edge data
df = result.to_pandas()
print("\nSample edges:")
print(df.head())

## Export Results

Export query results to various formats.

In [ ]:
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute("degree", "betweenness_centrality")
     .execute(network)
)

# Export to pandas DataFrame
df = result.to_pandas()
print("Pandas DataFrame:")
print(df.head())

# Export to dictionary
data_dict = result.to_dict()
print(f"\nDictionary with {len(data_dict)} entries")

# Export to NetworkX graph
nx_graph = result.to_networkx()
print(f"\nNetworkX graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")

## Group Results by Layer

Use `per_layer()` and `per_layer_pair()` to group results by layers.

In [ ]:
# Group nodes by layer and compute top-k per layer
result = (
    Q.nodes()
     .from_layers(L["*"])
     .compute("degree", "betweenness_centrality")
     .per_layer()  # Group by layer
        .top_k(3, "degree")  # Top 3 nodes per layer by degree
     .end_grouping()
     .execute(network)
)

print("\nTop 3 nodes per layer by degree:")
print(result.to_pandas())

# Get a summary of the grouping
print("\nGrouping summary:")
summary = result.group_summary()
print(summary)

## Uncertainty Quantification

Compute metrics with confidence intervals using bootstrap resampling.

In [ ]:
from py3plex.dsl import UQ

# Query with uncertainty quantification
result = (
    Q.nodes()
     .from_layers(L["*"])
     .uq(method="bootstrap", n_samples=50, ci=0.95)  # Bootstrap with 50 samples
     .compute("betweenness_centrality")
     .order_by("-betweenness_centrality")  # Sort by mean value
     .limit(5)
     .execute(network)
)

# Convert to DataFrame with expanded uncertainty columns
df = result.to_pandas(expand_uncertainty=True)
print("\nTop 5 nodes with uncertainty:")
print(df[['id', 'layer', 'betweenness_centrality', 
          'betweenness_centrality_std', 'betweenness_centrality_ci95_low', 
          'betweenness_centrality_ci95_high']])

## Temporal Queries

Query temporal networks with time windows (requires temporal network).

In [ ]:
from py3plex.core.temporal_multinet import TemporalMultiLayerNetwork

# Create a temporal network for demonstration
temporal_net = TemporalMultiLayerNetwork()
temporal_net.add_edge('A', 'B', layer='social', time=1.0)
temporal_net.add_edge('B', 'C', layer='social', time=2.0)
temporal_net.add_edge('A', 'C', layer='social', time=3.0)
temporal_net.add_edge('D', 'E', layer='social', time=4.0)

print("\nTemporal network created with 4 edges at different times")

# Query with time window
result = (
    Q.nodes()
     .from_layers(L["social"])
     .window(start=1.0, end=3.0)  # Only events between t=1.0 and t=3.0
     .compute("degree")
     .execute(temporal_net)
)

print("\nNodes active in time window [1.0, 3.0]:")
print(result.to_pandas())

## Summary

In this tutorial, you learned:

* ✅ String DSL syntax for quick queries
* ✅ Builder API (Q, L) for type-safe queries
* ✅ Layer algebra for combining layers
* ✅ Computing multiple metrics
* ✅ Complex filtering conditions
* ✅ Edge queries and analysis
* ✅ Grouping results by layer
* ✅ Uncertainty quantification with bootstrap
* ✅ Temporal queries with time windows
* ✅ Export options (pandas, JSON, CSV)

## Next Steps

* Check the [DSL reference](https://skblaz.github.io/py3plex/reference/dsl.html)
* Try [dynamics simulation](simulate_dynamics.ipynb)
* Explore [community detection](community_detection.ipynb)
* Read the [full documentation](https://skblaz.github.io/py3plex/)